**실행 설정**

입력은 `config/paths.local.json` 또는 `SLC7A5_DATA_ROOT`로 지정합니다. 결과는 이 저장소의 `results/`에 기록합니다. 입력/실행 설명은 저장소 루트 README를 참고하세요. 이 노트북에는 기존 탐색/대체 Figure 셀이 포함되어 있으며 전체 Figure 재현 검증은 별도 단계입니다.

**2026-10-06 갱신:** 구성 비교의 공식 검정은 donor별 비율의 exact permutation Mann–Whitney U + 9개 celltype BH 보정입니다. Healthy CD45 library는 donor별로 합치며 MASLD sample ID는 독립 donor로 가정합니다. 임상 donor 매핑은 검토해야 합니다. 기존 pooled Fisher/미보정 P값 별표 셀은 donor별 표와 boxplot으로 대체했습니다. DEG 등 다른 탐색 분석의 통계 단위까지 변경한 것은 아닙니다.


In [ ]:
# Repository paths; inputs are read-only, new files stay in results/.
from pathlib import Path
import os
import sys
_start = Path(os.environ.get("SLC7A5_REPO_ROOT", str(Path.cwd()))).resolve()
_repo = next((p for p in [_start, *_start.parents] if (p / "slc7a5_paths.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the repository or set SLC7A5_REPO_ROOT.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from slc7a5_paths import artifact_path, dataset_path, input_path, output_path, output_dir, setup_notebook
setup_notebook('figure2')


In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import seaborn as sns
import scipy.sparse as sp



In [ ]:
import matplotlib as mpl

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.use14corefonts"] = False

# Figure 2A

In [ ]:
adata=sc.read(artifact_path('MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono_cellannot.h5ad'))
adata

In [ ]:
adata.obs['state'].value_counts()

In [ ]:
count_table = pd.crosstab(adata.obs["celltype"], adata.obs["state"])
print(count_table)

In [ ]:
adata.write(output_path('02_SLC7A5_mono_Journal/01_Data/scRNA_adata_mono_cellannot.h5ad'))

In [ ]:
sc.pl.umap(
    adata,
    color='10_cluster',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2a_v1.pdf'), bbox_inches='tight')
plt.close()

In [ ]:
adata_filtered = adata[adata.obs['10_cluster'] != '6'].copy()
adata_filtered

In [ ]:
adata_filtered.write(output_path('MASLD_collaboration/multiomics_analysis/scRNA_v1/scRNA_adata_mono_cellannot_final.h5ad'))

In [ ]:
adata_filtered.write(output_path('02_SLC7A5_mono_Journal/01_Data/scRNA_adata_mono_cellannot_final.h5ad'))

In [ ]:
adata_filtered=sc.read(artifact_path('02_SLC7A5_mono_Journal/01_Data/scRNA_adata_mono_cellannot_final.h5ad'))
adata_filtered

In [ ]:
adata_filtered.obs['sample'] = adata_filtered.obs['sample'].replace({
    'Healthy1_Cd45p':'Healthy_1',
    'Healthy1_Cd45mA':'Healthy_1',
    'Healthy1_Cd45mB':'Healthy_1',
    'Healthy2_Cd45p':'Healthy_2',
    'Healthy2_Cd45m':'Healthy_2',
    'Healthy3_Cd45p':'Healthy_3',
    'Healthy3_Cd45mA':'Healthy_3',
    'Healthy3_Cd45mB':'Healthy_3',
    'Healthy4_Cd45m':'Healthy_4',
    'Healthy4_Cd45p':'Healthy_4',
    'Healthy5_Cd45p':'Healthy_5'
})

sc.pl.umap(adata_filtered, color=['sample'])

In [ ]:
sc.pl.umap(
    adata_filtered,
    color='10_cluster',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2a_v2.pdf'), bbox_inches='tight')
plt.close()

In [ ]:
sc.pl.umap(
    adata_filtered,
    color='monocluster',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = adata_filtered.obs['monocluster']
batch_labels = adata_filtered.obs['sample']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(12, 5))

# 그래프 설정
plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASL vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/monomac_state_stackedbargraph.pdf'), bbox_inches='tight', dpi=300)

In [ ]:
adata_filtered.obs['state'] = adata_filtered.obs['state'].replace({
    'Healthy': 'Healthy',
    'MASL': 'MASLD'})

In [ ]:
sc.pl.umap(adata_filtered, color=['state'])

In [ ]:
adata_filtered.obs['celltype'] = adata_filtered.obs['monocluster']
adata_filtered.obs['celltype'] = adata_filtered.obs['celltype'].replace({
    'mono0': 'Classical_monocyte',
    'mono1': 'Kupffer_cell',
    'mono2': 'cDC2',
    'mono3': 'Non-classical_monocyte',
    'mono4': 'LAM',
    'mono5': 'Special_monocyte',
    'mono7': 'mo_Kupffer_cell',
    'mono8': 'cDC1',
    'mono9': 'mig_DC'
})

sc.pl.umap(adata_filtered, color=['celltype'])

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = adata_filtered.obs['celltype']
batch_labels = adata_filtered.obs['sample']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(12, 5))

# 그래프 설정
plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASL vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/celltype_state_stackedbargraph.pdf'), bbox_inches='tight', dpi=300)

In [ ]:
# Donor-level celltype proportions: Healthy vs MASLD
# One observation per donor; denominator = all cells in adata_filtered (myeloid).
# Healthy CD45 libraries are merged. MASLD sample IDs are assumed to be distinct
# donors; edit fig2_donor_overrides if clinical metadata specifies another mapping.
# Test: two-sided exact permutation of the Mann-Whitney U statistic on donor
# proportions, followed by BH correction across the displayed celltypes.
# This test is NOT the beta-binomial model described in the manuscript Methods.
from pathlib import Path
from math import comb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import permutation_test, rankdata
from statsmodels.stats.multitest import multipletests
from IPython.display import display

fig2_donor_overrides = {}  # e.g. {"sample_library_ID": "biological_donor_ID"}
fig2_donor_obs = adata_filtered.obs[["sample", "state", "celltype"]].copy()
if fig2_donor_obs.empty or fig2_donor_obs.isna().any().any():
    raise ValueError("sample, state and celltype must be present for every cell.")
fig2_donor_obs = fig2_donor_obs.astype(str)
fig2_donor_obs["state"] = fig2_donor_obs["state"].replace({"MASL": "MASLD"})
fig2_state_order = ["Healthy", "MASLD"]
if set(fig2_donor_obs["state"]) != set(fig2_state_order):
    raise ValueError("Both Healthy and MASLD must be present; check state labels.")
fig2_donor_obs["donor"] = fig2_donor_obs["sample"].str.replace(
    r"^Healthy_?(\d+)(?:_Cd45.*)?$", r"Healthy_\1", regex=True
)
fig2_donor_obs["donor"] = fig2_donor_obs["sample"].map(
    fig2_donor_overrides
).fillna(fig2_donor_obs["donor"])
if fig2_donor_obs.groupby("donor")["state"].nunique().gt(1).any():
    raise ValueError("A donor maps to multiple states; check the donor mapping.")

fig2_celltype_order = [
    "Classical_monocyte", "Non-classical_monocyte", "Special_monocyte",
    "LAM", "Kupffer_cell", "mo_Kupffer_cell", "cDC1", "cDC2", "mig_DC",
]
if set(fig2_donor_obs["celltype"]) - set(fig2_celltype_order):
    raise ValueError("Run the celltype annotation cell before this cell.")
fig2_donor_meta = fig2_donor_obs.groupby("donor", sort=True).agg(
    state=("state", "first"),
    sample_ids=("sample", lambda values: ";".join(sorted(set(values)))),
    n_total=("celltype", "size"),
)
fig2_donor_counts = pd.crosstab(
    fig2_donor_obs["donor"], fig2_donor_obs["celltype"]
).reindex(index=fig2_donor_meta.index, columns=fig2_celltype_order, fill_value=0)
# Retain every donor, including donors with zero cells of a particular type.
fig2_donor_freq = fig2_donor_counts.rename_axis(columns="celltype").reset_index().melt(
    id_vars="donor", var_name="celltype", value_name="n_celltype"
).merge(fig2_donor_meta.reset_index(), on="donor", validate="many_to_one")
fig2_donor_freq["fraction"] = fig2_donor_freq["n_celltype"] / fig2_donor_freq["n_total"]
fig2_donor_freq["percent"] = 100 * fig2_donor_freq["fraction"]
fig2_donor_n = fig2_donor_meta["state"].value_counts().reindex(fig2_state_order)
if fig2_donor_n.min() < 2:
    raise ValueError("At least two independent donors per group are required.")

def fig2_donor_u(x, y):
    ranks = rankdata(np.concatenate([x, y]), method="average")
    return ranks[:len(x)].sum() - len(x) * (len(x) + 1) / 2

# For the current 5 + 6 donors, enumerate all 462 possible group assignments.
fig2_n_assignments = comb(int(fig2_donor_n.sum()), int(fig2_donor_n["Healthy"]))
fig2_n_resamples = np.inf if fig2_n_assignments <= 100000 else 99999
fig2_test_mode = "exact" if np.isinf(fig2_n_resamples) else "Monte Carlo"
fig2_donor_rows = []
for fig2_ct in fig2_celltype_order:
    fig2_sub = fig2_donor_freq.loc[fig2_donor_freq["celltype"].eq(fig2_ct)]
    fig2_h = fig2_sub.loc[fig2_sub["state"].eq("Healthy"), "percent"].to_numpy()
    fig2_m = fig2_sub.loc[fig2_sub["state"].eq("MASLD"), "percent"].to_numpy()
    fig2_test = permutation_test(
        (fig2_h, fig2_m), fig2_donor_u, permutation_type="independent",
        vectorized=False, n_resamples=fig2_n_resamples,
        alternative="two-sided", random_state=42,
    )
    fig2_donor_rows.append({
        "celltype": fig2_ct,
        "n_Healthy_donors": len(fig2_h), "n_MASLD_donors": len(fig2_m),
        "Healthy_median_pct": np.median(fig2_h), "MASLD_median_pct": np.median(fig2_m),
        "median_difference_pp_MASLD_minus_Healthy": np.median(fig2_m) - np.median(fig2_h),
        "U_Healthy": fig2_test.statistic, "p_value": fig2_test.pvalue,
        "test": f"Two-sided donor-level Mann-Whitney U, {fig2_test_mode} permutations",
        "denominator": "All myeloid cells in adata_filtered per donor",
    })
fig2_donor_stats = pd.DataFrame(fig2_donor_rows)
fig2_donor_stats["q_value_BH"] = multipletests(
    fig2_donor_stats["p_value"], method="fdr_bh"
)[1]

fig2_donor_root = Path(output_dir('02_SLC7A5_mono_Journal'))
fig2_donor_figdir = fig2_donor_root / "05_Figure"
fig2_donor_outdir = fig2_donor_root / "02_Analysis/Fig2_donor_level_composition"
fig2_donor_figdir.mkdir(parents=True, exist_ok=True)
fig2_donor_outdir.mkdir(parents=True, exist_ok=True)
fig2_donor_meta.to_csv(fig2_donor_outdir / "donor_metadata.csv")
fig2_donor_freq.to_csv(fig2_donor_outdir / "donor_celltype_frequencies.csv", index=False)
fig2_donor_stats.to_csv(fig2_donor_outdir / "donor_celltype_statistics.csv", index=False)

fig2_donor_palette = {"Healthy": "#4C78A8", "MASLD": "#E07B54"}
fig2_donor_titles = {
    "Classical_monocyte": "Classical monocyte",
    "Non-classical_monocyte": "Non-classical monocyte",
    "Special_monocyte": "SLC7A5-high monocyte",
    "Kupffer_cell": "Kupffer cell", "mo_Kupffer_cell": "Monocyte-derived Kupffer cell",
    "mig_DC": "Migratory DC",
}

def fig2_plot_donor_box(ax, celltype):
    values = fig2_donor_freq.loc[fig2_donor_freq["celltype"].eq(celltype)]
    result = fig2_donor_stats.set_index("celltype").loc[celltype]
    sns.boxplot(
        data=values, x="state", y="percent", order=fig2_state_order,
        palette=fig2_donor_palette, width=0.52, showfliers=False,
        linewidth=1.2, saturation=0.75, ax=ax,
    )
    # Deterministic jitter; use the same donor positions in every panel.
    rng = np.random.default_rng(42)
    for xpos, state in enumerate(fig2_state_order):
        points = values.loc[values["state"].eq(state)].sort_values("donor")
        ax.scatter(
            xpos + rng.permutation(np.linspace(-0.16, 0.16, len(points))), points["percent"],
            s=40, c=fig2_donor_palette[state], edgecolors="black",
            linewidths=0.65, zorder=3,
        )
    top = max(float(values["percent"].max()), 1.0)
    bracket_y, bracket_h = top * 1.10, top * 0.04
    ax.plot([0, 0, 1, 1], [bracket_y, bracket_y + bracket_h,
            bracket_y + bracket_h, bracket_y], color="0.25", lw=1)
    ax.text(
        0.5, bracket_y + 1.4 * bracket_h,
        f"p = {result['p_value']:.3g}; BH q = {result['q_value_BH']:.3g}",
        ha="center", va="bottom", fontsize=9,
    )
    ax.set_ylim(-top * 0.04, top * 1.32)
    ax.set_xticks([0, 1], [f"{state}\n(n = {fig2_donor_n[state]} donors)"
                          for state in fig2_state_order])
    ax.set_xlabel("")
    ax.set_ylabel("Cells / donor myeloid cells (%)")
    ax.set_title(fig2_donor_titles.get(celltype, celltype), fontsize=11, fontweight="bold")
    sns.despine(ax=ax)

with plt.rc_context({"pdf.fonttype": 42, "ps.fonttype": 42, "font.family": "DejaVu Sans"}):
    fig2_donor_fig, fig2_donor_axes = plt.subplots(3, 3, figsize=(13, 12))
    for fig2_ax, fig2_ct in zip(fig2_donor_axes.flat, fig2_celltype_order):
        fig2_plot_donor_box(fig2_ax, fig2_ct)
    fig2_donor_fig.suptitle("Donor-level myeloid composition: Healthy vs MASLD", fontsize=15)
    fig2_donor_fig.text(
        0.5, 0.01,
        f"Each point = one donor | Two-sided {fig2_test_mode} permutation of Mann-Whitney U | BH across 9 celltypes",
        ha="center", fontsize=10,
    )
    fig2_donor_fig.tight_layout(rect=[0, 0.04, 1, 0.96])
    for fig2_ext in ("pdf", "png"):
        fig2_donor_fig.savefig(fig2_donor_figdir / f"Fig2_donor_celltype_boxplots.{fig2_ext}",
                              bbox_inches="tight", dpi=300)
    plt.show()
    plt.close(fig2_donor_fig)

    fig2_high_fig, fig2_high_ax = plt.subplots(figsize=(5.5, 5.7))
    fig2_plot_donor_box(fig2_high_ax, "Special_monocyte")
    fig2_high_fig.text(
        0.5, 0.01,
        f"Two-sided {fig2_test_mode} permutation of Mann-Whitney U\nEach point = one donor; BH q across 9 celltypes",
        ha="center", fontsize=8,
    )
    fig2_high_fig.tight_layout(rect=[0, 0.06, 1, 1])
    for fig2_ext in ("pdf", "png"):
        fig2_high_fig.savefig(fig2_donor_figdir / f"Fig2_donor_SLC7A5high_boxplot.{fig2_ext}",
                             bbox_inches="tight", dpi=300)
    plt.show()
    plt.close(fig2_high_fig)

print("Healthy CD45 libraries merged; MASLD sample IDs assumed to identify distinct donors.")
print(f"Statistics: {fig2_test_mode} donor-level permutations; not a beta-binomial model.")
display(fig2_donor_stats[["celltype", "n_Healthy_donors", "n_MASLD_donors",
                         "Healthy_median_pct", "MASLD_median_pct", "p_value", "q_value_BH"]])


In [ ]:
sc.pl.umap(
    adata_filtered,
    color='celltype',
    legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10,
    show=False)
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2Av1.pdf'), bbox_inches='tight')
plt.close()

In [ ]:
sc.pl.umap(
    adata_filtered,
    color='celltype',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10,
    show=False)
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2Av2.pdf'), bbox_inches='tight')
plt.close()

In [ ]:
sc.tl.rank_genes_groups(
    adata_filtered,
    groupby='state',
    method='wilcoxon',
    use_raw=True,
    pts=True
)

sc.pl.rank_genes_groups_dotplot(adata_filtered, n_genes=5)
deg_all = sc.get.rank_genes_groups_df(adata_filtered, group=None)
deg_all

In [ ]:
top_genes = deg_all.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_filtered,
    var_names=top_genes,
    groupby="state",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True, 
    #return_fig=True,
    figsize=(12, 7)
)
# 직접 저장
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_heatmap_Fig2B_v1.pdf'), bbox_inches='tight', dpi=300)
# /home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/total_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
# RPS, RPL, MT 제외 필터
top_genes = (
    deg_all
    .groupby("group")["names"]
    .apply(lambda x: x.head(100))  # 넉넉하게 뽑아서 필터 후 20개 확보
    .explode()
    .reset_index(drop=True)
)

filtered_genes = top_genes[
    ~top_genes.str.startswith(("RPS", "RPL", "MT-"))
]

top_genes_final = (
    deg_all[deg_all["names"].isin(filtered_genes)]
    .groupby("group")["names"]
    .apply(lambda x: x.head(20))
    .explode()
    .unique()
    .tolist()
)

sc.pl.heatmap(
    adata_filtered,
    var_names=top_genes_final,
    groupby="state",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(12, 7)
)

plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_heatmap_Fig2B_v2.pdf'), bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
deg_all.to_csv(output_path('02_SLC7A5_mono_Journal/05_Figure/scRNA_monomac_state_DEG.csv'))

In [ ]:
sc.tl.rank_genes_groups(
    adata_filtered,
    groupby='monocluster',
    method='wilcoxon',
    use_raw=True,
    pts=True
)

sc.pl.rank_genes_groups_dotplot(adata_filtered, n_genes=5)
deg_mono = sc.get.rank_genes_groups_df(adata_filtered, group=None)
deg_mono

In [ ]:
top_genes = deg_mono.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_filtered,
    var_names=top_genes,
    groupby="monocluster",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True, 
    #return_fig=True,
    figsize=(12, 7)
)
# 직접 저장
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_heatmap_Fig2C_v1.pdf'), bbox_inches='tight', dpi=300)
# /home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/total_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
deg_mono.to_csv(output_path('02_SLC7A5_mono_Journal/05_Figure/scRNA_monomac_monocluster_DEG.csv'))

# Figure 2B

In [ ]:
adata

In [ ]:
adata_filtered=sc.read(artifact_path('02_SLC7A5_mono_Journal/01_Data/scRNA_adata_mono_cellannot_final.h5ad'))
adata_filtered

In [ ]:
adata_filtered.obs['monocluster']

In [ ]:
adata_filtered.obs['celltype'] = adata_filtered.obs['monocluster']
adata_filtered.obs['celltype'] = adata_filtered.obs['celltype'].replace({
    'mono0': 'Classical_monocyte',
    'mono1': 'Kupffer_cell',
    'mono2': 'cDC2',
    'mono3': 'Non-classical_monocyte',
    'mono4': 'LAM',
    'mono5': 'Special_monocyte',
    'mono7': 'mo_Kupffer_cell',
    'mono8': 'cDC1',
    'mono9': 'mig_DC'
})

sc.pl.umap(adata_filtered, color=['celltype'])

In [ ]:
desired_order = ['Classical_monocyte','Non-classical_monocyte','Special_monocyte','LAM','Kupffer_cell','mo_Kupffer_cell','cDC1','cDC2','mig_DC']
adata_filtered.obs['celltype'] = adata_filtered.obs['celltype'].astype('category')

adata_filtered.obs['celltype'] = (
    adata_filtered.obs['celltype']
    .cat.reorder_categories(desired_order, ordered=True)
)

In [ ]:
sc.pl.umap(
    adata_filtered,
    color='celltype',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10,
    show=False)
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2Av3.pdf'), bbox_inches='tight')
plt.close()

In [ ]:
sc.pl.umap(
    adata_filtered,
    color='celltype',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type annotation',
    size=10
)
#plt.savefig('/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/Fig2a_v3.pdf', bbox_inches='tight')
plt.close()

In [ ]:
sc.tl.rank_genes_groups(
    adata_filtered,
    groupby='celltype',
    method='wilcoxon',
    use_raw=True,
    pts=True
)

sc.pl.rank_genes_groups_dotplot(adata_filtered, n_genes=5)
deg_mono = sc.get.rank_genes_groups_df(adata_filtered, group=None)
deg_mono

In [ ]:
top_genes = deg_mono.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_filtered,
    var_names=top_genes,
    groupby="celltype",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True, 
    #return_fig=True,
    figsize=(12, 7)
)
# 직접 저장
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_heatmap_Fig2C_v2.pdf'), bbox_inches='tight', dpi=300)
# /home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/total_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
top_genes = deg_mono.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_filtered,
    var_names=top_genes,
    groupby="celltype",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True, 
    #return_fig=True,
    figsize=(12, 7)
)
# 직접 저장
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_heatmap_Fig2C_v3.pdf'), bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
# 그룹별 샘플링 수
n_cells_per_group = 100
groups = adata_filtered.obs['celltype'].unique()

# 샘플링된 셀들의 인덱스 저장
selected_indices = []

for group in groups:
    group_cells = adata_filtered.obs[adata_filtered.obs['celltype'] == group].index
    n = min(len(group_cells), n_cells_per_group)  # 그룹 크기가 작을 경우 자동 조절
    sampled = np.random.choice(group_cells, n, replace=False)
    selected_indices.extend(sampled)

adata_sampled = adata_filtered[selected_indices]

In [ ]:
fig_dict = sc.pl.rank_genes_groups_heatmap(
    adata_sampled,
    n_genes=10,
    groupby="celltype",
    use_raw=False,
    swap_axes=True,
    vmin=-3,
    vmax=3,
    dendrogram=False,
    figsize=(10, 7),
    cmap="bwr",
    show=False,
    show_gene_labels=False,
    return_fig=True
)
fig = fig_dict['heatmap_ax'].figure

fig.savefig(
    output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_heatmap_Fig2C_v4.pdf'),
    bbox_inches='tight',
    dpi=300
)

In [ ]:
fig_dict = sc.pl.rank_genes_groups_heatmap(
    adata_sampled,
    n_genes=10,
    groupby="celltype",
    use_raw=False,
    swap_axes=True,
    vmin=-3,
    vmax=3,
    dendrogram=False,
    figsize=(10, 12),      # 높이 7 → 12으로 키움 (유전자 라벨 공간 확보)
    cmap="bwr",
    show=False,
    show_gene_labels=True, # False → True
    return_fig=True
)

fig = fig_dict['heatmap_ax'].figure

# 유전자 라벨 폰트 크기 조정 (선택)
heatmap_ax = fig_dict['heatmap_ax']
heatmap_ax.tick_params(axis='y', labelsize=7)

#fig.savefig(
#    "/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/05_Figure/cellseq_heatmap_Fig2C_v4.pdf",
#    bbox_inches='tight',
#    dpi=300
#)

In [ ]:
# 각 celltype별 top 3 + Special Monocyte에 SLC7A5 추가
marker_genes_dict = {
    "Classical Monocyte":          ["S100A9", "S100A8", "S100A6","LYZ","FCN1"],
    "Non-classical Monocyte":      ["IFITM2","IFITM3", "CD52", "LST1","FCGR3A"],
    "Special Monocyte":            ["PPIF", "ANPEP", "SLC7A5","ITGAX","VCAN"],
    "LAM":                         ["TREM2","LGALS3","C1QC", "APOC1", "GPNMB"],
    "Kupffer Cell":                 ["CD5L", "CD163", "SLC40A1","MARCO","VCAM1"],
    "Monocyte-derived Kupffer Cell": ["FTL", "C1QA","C1QB", "AIF1","FTH1"],
    "cDC1":                        ["SNX3", "DNASE1L3", "CLEC9A","IRF8","IDO1"],
    "cDC2":                        ["HLA-DQB1", "HLA-DPB1", "HLA-DQA1","CD1C","FCER1A"],
    "Migratory DC":                ["GBP1", "IL4I1", "CXCL10","LAMP3","CD83"],
}

# heatmap 그리기
fig_dict = sc.pl.rank_genes_groups_heatmap(
    adata_sampled,
    var_names=marker_genes_dict,   # dict로 넘기면 그룹 라벨도 자동 표시
    groupby="celltype",
    use_raw=False,
    swap_axes=True,
    vmin=-3,
    vmax=3,
    dendrogram=False,
    figsize=(10, 7),
    cmap="bwr",
    show=False,
    show_gene_labels=True,
    return_fig=True
)

# y축 유전자 라벨 볼드 처리 + SLC7A5 강조
heatmap_ax = fig_dict['heatmap_ax']

for label in heatmap_ax.get_yticklabels():
    gene = label.get_text()
    label.set_fontweight('bold')
    label.set_fontsize(8)
    if gene == 'SLC7A5':
        label.set_color('#CC0000')   # SLC7A5만 빨간 볼드로 강조

fig = heatmap_ax.figure
fig.savefig(
    output_path('02_SLC7A5_mono_Journal/05_Figure/cellseq_heatmap_Fig2C_v5.pdf'),
    bbox_inches='tight',
    dpi=300
)

In [ ]:
print(adata_sampled.obs['celltype'].cat.categories)
print(set(adata_sampled.obs['celltype'].cat.categories) - set(desired_order))
print(set(desired_order) - set(adata_sampled.obs['celltype'].cat.categories))

In [ ]:
sc.pl.umap(
    adata_filtered,
    color='celltype',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type annotation',
    size=10
)

In [ ]:
celltypes = adata_filtered.obs['celltype'].cat.categories
palette_dict = dict(zip(celltypes, adata_filtered.uns['celltype_colors']))
# 2. crosstab 생성
cluster_labels = adata_filtered.obs['celltype']
batch_labels = adata_filtered.obs['state']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab = crosstab.reindex(['Healthy', 'MASL'])
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2B_stackedbargraph.pdf'), bbox_inches='tight', dpi=300)

In [ ]:
celltypes = adata_filtered.obs['celltype'].cat.categories
palette_dict = dict(zip(celltypes, adata_filtered.uns['celltype_colors']))

cluster_labels = adata_filtered.obs['celltype']
batch_labels = adata_filtered.obs['state']

crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')
crosstab = crosstab.reindex(['Healthy', 'MASL'])

crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD vs Normal')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD vs Normal', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig(
    output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2B_stackedbargraph.pdf'),
    bbox_inches='tight',
    dpi=300
)

In [ ]:
# Official composition inference uses independent donors (not pooled Fisher tests).
# The earlier donor block calculates exact permutation Mann-Whitney U and BH q-values.
df_result = fig2_donor_stats.copy()
display(df_result)


In [ ]:
# Updated composition panel: one point per donor, BH correction across nine celltypes.
fig, axes = plt.subplots(3, 3, figsize=(13, 12))
for ax, celltype in zip(axes.flat, fig2_celltype_order):
    fig2_plot_donor_box(ax, celltype)
fig.suptitle('Donor-level myeloid composition: Healthy vs MASLD')
fig.text(0.5, 0.015, 'One point per donor | Exact permutation Mann-Whitney U | BH across 9 celltypes', ha='center')
fig.tight_layout(rect=[0, 0.04, 1, 0.96])
fig.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2B_proportionbargraph.pdf'), bbox_inches='tight', dpi=300)
plt.show()
plt.close(fig)


In [ ]:
genes = ['SLC7A5','CD14', 'FCGR3A']
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(10, 3 * n_genes))

# genes가 1개일 때 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_filtered,
        keys=gene,
        groupby='celltype',
        ax=axs[i],
        use_raw=False,
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)
    plt.setp(axs[i].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2C_violinplot.pdf'), bbox_inches='tight', dpi=300)

In [ ]:
genes = ['TLR4','LYN','CXCL8','SYK','BCL2A1','IL1R1','PRKCB','TNFAIP3',
         'CXCL3','TICAM1','CXCL2','MALT1','RELA','NFKB1','RELB','NFKB2','TNFRSF1A',
         'IRAK1','TRAF3','TRIM25','TLR4','MYD88','BIRC2']
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(10, 3 * n_genes))

# genes가 1개일 때 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_filtered,
        keys=gene,
        groupby='celltype',
        ax=axs[i],
        use_raw=False,
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)
    plt.setp(axs[i].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2C_violinplot_NFKB.pdf'), bbox_inches='tight', dpi=300)

In [ ]:
adata=sc.read(artifact_path('02_SLC7A5_mono_Journal/01_Data/scRNA_adata_mono_cellannot_final.h5ad'))

sc.tl.rank_genes_groups(
    adata,
    groupby='celltype',
    method='wilcoxon',
    use_raw=True,
    pts=True
)

sc.pl.rank_genes_groups_dotplot(adata, n_genes=5)
deg_cell = sc.get.rank_genes_groups_df(adata, group=None)
deg_cell

In [ ]:
desired_order = ['Classical_monocyte','Non-classical_monocyte','Special_monocyte','LAM','Kupffer_cell','mo_Kupffer_cell','cDC1','cDC2','mig_DC']
adata.obs['celltype'] = adata.obs['celltype'].astype('category')

adata.obs['celltype'] = (
    adata.obs['celltype']
    .cat.reorder_categories(desired_order, ordered=True)
)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Dotplot (ALL clusters from deg_cell) — Mean Expression + %Expressing
# ──────────────────────────────────────────────────────────────────────────

target_genes = ['TLR4','LYN','CXCL8','SYK','BCL2A1','IL1R1','PRKCB','TNFAIP3',
         'CXCL3','TICAM1','CXCL2','MALT1','RELA','NFKB1','RELB','NFKB2','TNFRSF1A',
         'IRAK1','TRAF3','TRIM25','MYD88','BIRC2']
clusters = [cl for cl in desired_order if cl in deg_cell['group'].unique()]

# cluster별 deg table dict화
deg_dict = {
    cl: deg_cell[deg_cell['group'] == cl].set_index('names')
    for cl in clusters
}

def get_raw_expr(adata, gene):
    if adata.raw is None:
        raise ValueError("adata.raw is None")
    if gene not in adata.raw.var_names:
        raise ValueError(f"{gene} not found in raw")
    idx = np.where(adata.raw.var_names == gene)[0][0]
    expr = adata.raw.X[:, idx]
    return expr.toarray().flatten() if sp.issparse(expr) else expr.flatten()

# ── 데이터 수집 ────────────────────────────────────────────────────────────
rows_g = []
for gene in target_genes:
    for cl in clusters:
        cl_label = f'Cluster {cl}'
        deg_sub = deg_dict[cl]

        # pct_nz_group
        pct_val = np.nan
        if gene in deg_sub.index and 'pct_nz_group' in deg_sub.columns:
            pct_val = float(deg_sub.loc[gene, 'pct_nz_group']) * 100

        # raw mean expression
        mask = adata.obs['celltype'] == cl
        if gene in adata.raw.var_names:
            expr_vals = get_raw_expr(adata, gene)[mask]
        else:
            expr_vals = np.zeros(mask.sum())
        logfc = np.nan
        if gene in deg_sub.index and 'logfoldchanges' in deg_sub.columns:
            logfc = float(deg_sub.loc[gene, 'logfoldchanges'])

        rows_g.append({
            'gene': gene,
            'cluster': cl,
            'cluster_id': cl,
            'mean_expr': float(expr_vals.mean()),
            'pct': pct_val,
            'logFC': logfc 
        })

df_g = pd.DataFrame(rows_g)
df_g['gene'] = pd.Categorical(df_g['gene'], categories=target_genes, ordered=True)

# ── Figure: Dotplot (axes swapped) ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(len(clusters) * 0.9 + 2, len(target_genes) * 0.9))

cluster_order = [f'{cl}' for cl in clusters]
gene_order = target_genes

cmap = plt.cm.Reds
vmin = 0
vmax = 2

for yi, gene in enumerate(gene_order):           # ← y: gene
    for xi, cl in enumerate(cluster_order):      # ← x: cluster
        row = df_g[(df_g['gene'] == gene) & (df_g['cluster'] == cl)]
        if len(row) == 0:
            continue

        pct_val = float(row['pct'].values[0]) if not pd.isna(row['pct'].values[0]) else 0.0
        #expr_val = float(row['mean_expr'].values[0]
        logfc_val = float(row['logFC'].values[0]) if not pd.isna(row['logFC'].values[0]) else 0.0
        logfc_val = np.clip(logfc_val, 0, 2)

        size = max(4, (pct_val / 100) * 800 * 1.2)
        alpha = 0.85 if pct_val > 0 else 0.2
        #color = cmap((expr_val - vmin) / (vmax - vmin + 1e-9))
        norm_val = (logfc_val - vmin) / (vmax - vmin + 1e-9)
        color = cmap(norm_val)

        ax.scatter(
            xi, yi,
            s=size,
            c=[color],
            edgecolors='none',
            linewidths=0.5,
            alpha=alpha,
            zorder=3
        )

       # ax.text(
       #     xi + 0.35, yi,                     # ← 핵심 수정
       #     f'{pct_val:.1f}%',
       #     ha='left',
       #     va='center',
       #     fontsize=8,
       #     color='#333333'
       # )

# 축 설정 (swap 반영)
ax.set_xticks(range(len(cluster_order)))
ax.set_xticklabels(cluster_order, fontsize=10, rotation=45, ha='right')

ax.set_yticks(range(len(gene_order)))
ax.set_yticklabels(gene_order, fontsize=11, style='italic')

ax.set_xlim(-0.6, len(cluster_order) - 0.2)  # 오른쪽 공간 약간 확장
ax.set_ylim(-0.6, len(gene_order) - 0.4)
ax.invert_yaxis()

ax.set_title(
    'Amino acid transporter gene dotplot\n'
    '(Dot size = %Expressing, Dot color = Log2FC)',
    fontsize=11
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# colorbar
sm = plt.cm.ScalarMappable(
    cmap=cmap,
    norm=plt.Normalize(vmin=vmin, vmax=vmax)
)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label('log2 Fold Change (0–2)', fontsize=9)

# size legend
import matplotlib.lines as mlines
leg_handles = [
    mlines.Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='#888888',
        markeredgecolor='grey',
        markersize=np.sqrt((pct_leg / 100) * 800),
        label=f'{pct_leg}%',
        alpha=0.7
    )
    for pct_leg in [10, 30, 50]
]
ax.grid(False)
ax.legend(
    handles=leg_handles,
    title='% Expressing',
    loc='lower right',
    fontsize=8,
    title_fontsize=8,
    framealpha=0.4
)

plt.tight_layout()
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2C_bubbleplot_NFKB.pdf'), bbox_inches='tight', dpi=300)
plt.show()

pt = df_g.pivot_table(
    index='gene',
    columns='cluster',
    values=['mean_expr', 'pct'],
    sort=False
)

pt = pt.reindex(target_genes)

print(pt.round(3).to_string())

In [ ]:
# Dotplot (ALL clusters from deg_cell) — Mean Expression + %Expressing + adj.P star

target_genes = [
    'TLR4','LYN','CXCL8','SYK','BCL2A1','IL1R1','PRKCB','TNFAIP3',
    'CXCL3','TICAM1','CXCL2','MALT1','RELA','NFKB1','RELB','NFKB2',
    'TNFRSF1A','IRAK1','TRAF3','TRIM25','MYD88','BIRC2'
]

clusters = [cl for cl in desired_order if cl in deg_cell['group'].unique()]

deg_dict = {
    cl: deg_cell[deg_cell['group'] == cl].set_index('names')
    for cl in clusters
}

def get_raw_expr(adata, gene):
    if adata.raw is None:
        raise ValueError("adata.raw is None")
    if gene not in adata.raw.var_names:
        raise ValueError(f"{gene} not found in raw")
    idx = np.where(adata.raw.var_names == gene)[0][0]
    expr = adata.raw.X[:, idx]
    return expr.toarray().flatten() if sp.issparse(expr) else expr.flatten()

rows_g = []
for gene in target_genes:
    for cl in clusters:
        deg_sub = deg_dict[cl]

        pct_val = np.nan
        if gene in deg_sub.index and 'pct_nz_group' in deg_sub.columns:
            pct_val = float(deg_sub.loc[gene, 'pct_nz_group']) * 100

        mask = adata.obs['celltype'] == cl
        if gene in adata.raw.var_names:
            expr_vals = get_raw_expr(adata, gene)[mask]
        else:
            expr_vals = np.zeros(mask.sum())

        logfc = np.nan
        if gene in deg_sub.index and 'logfoldchanges' in deg_sub.columns:
            logfc = float(deg_sub.loc[gene, 'logfoldchanges'])

        p_adj = np.nan
        if gene in deg_sub.index and 'pvals_adj' in deg_sub.columns:
            p_adj = float(deg_sub.loc[gene, 'pvals_adj'])

        rows_g.append({
            'gene': gene,
            'cluster': cl,
            'cluster_id': cl,
            'mean_expr': float(expr_vals.mean()) if len(expr_vals) > 0 else 0.0,
            'pct': pct_val,
            'logFC': logfc,
            'pvals_adj': p_adj
        })

df_g = pd.DataFrame(rows_g)
df_g['gene'] = pd.Categorical(df_g['gene'], categories=target_genes, ordered=True)
df_g['cluster'] = pd.Categorical(df_g['cluster'], categories=clusters, ordered=True)

fig, ax = plt.subplots(figsize=(len(clusters) * 0.9 + 2, len(target_genes) * 0.9))

cluster_order = clusters
gene_order = target_genes

cmap = plt.cm.Reds
vmin = 0
vmax = 2

for yi, gene in enumerate(gene_order):
    for xi, cl in enumerate(cluster_order):
        row = df_g[(df_g['gene'] == gene) & (df_g['cluster'] == cl)]
        if len(row) == 0:
            continue

        pct_val = float(row['pct'].values[0]) if not pd.isna(row['pct'].values[0]) else 0.0
        logfc_val = float(row['logFC'].values[0]) if not pd.isna(row['logFC'].values[0]) else 0.0
        p_adj_val = float(row['pvals_adj'].values[0]) if not pd.isna(row['pvals_adj'].values[0]) else np.nan

        logfc_val = np.clip(logfc_val, 0, 2)
        size = max(4, (pct_val / 100) * 800 * 1.2)
        alpha = 0.85 if pct_val > 0 else 0.2
        norm_val = (logfc_val - vmin) / (vmax - vmin + 1e-9)
        color = cmap(norm_val)

        ax.scatter(
            xi,
            yi,
            s=size,
            c=[color],
            edgecolors='none',
            linewidths=0,
            alpha=alpha,
            zorder=3
        )
        if (
            not np.isnan(p_adj_val)
            and p_adj_val < 0.05
            and not np.isnan(logfc_val)
            and logfc_val > 0.58
        ):
            ax.text(
                xi,
                yi,
                '*',
                ha='center',
                va='center',
                fontsize=10,
                color='black',
                fontweight='bold',
                zorder=4
            )
ax.set_xticks(range(len(cluster_order)))
ax.set_xticklabels(cluster_order, fontsize=10, rotation=45, ha='right')

ax.set_yticks(range(len(gene_order)))
ax.set_yticklabels(gene_order, fontsize=11, style='italic')

ax.set_xlim(-0.6, len(cluster_order) - 0.4)
ax.set_ylim(-0.6, len(gene_order) - 0.4)
ax.invert_yaxis()

ax.set_title(
    'NFKB pathway gene dotplot\n'
    '(Dot size = %Expressing, Dot color = Log2FC, * = adj.P ≤ 0.05)',
    fontsize=11
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

sm = plt.cm.ScalarMappable(
    cmap=cmap,
    norm=plt.Normalize(vmin=vmin, vmax=vmax)
)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label('log2 Fold Change (0–2)', fontsize=9)

import matplotlib.lines as mlines
leg_handles = [
    mlines.Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='#888888',
        markeredgecolor='none',
        markersize=np.sqrt((pct_leg / 100) * 800),
        label=f'{pct_leg}%',
        alpha=0.7
    )
    for pct_leg in [10, 30, 50]
]

ax.grid(False)
ax.legend(
    handles=leg_handles,
    title='% Expressing',
    loc='lower right',
    fontsize=8,
    title_fontsize=8,
    framealpha=0.4
)

plt.tight_layout()
plt.savefig(
    output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2C_bubbleplot_NFKBv2.pdf'),
    bbox_inches='tight',
    dpi=300
)
plt.show()

pt = df_g.pivot_table(
    index='gene',
    columns='cluster',
    values=['mean_expr', 'pct', 'logFC', 'pvals_adj'],
    sort=False
)
pt = pt.reindex(target_genes)

print(pt.round(3).to_string())

In [ ]:
genes = ['TREM2','CD9','LGALS3', 'PLIN2', 'GPNMB', 'SPP1','ACTA2','POSTN']
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(10, 3 * n_genes))

# genes가 1개일 때 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_filtered,
        keys=gene,
        groupby='celltype',
        ax=axs[i],
        use_raw=False,
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)
    plt.setp(axs[i].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
genes = ['CD33','ITGAM', 'OLR1','ARG1','S100A8']
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(10, 3 * n_genes))

# genes가 1개일 때 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_filtered,
        keys=gene,
        groupby='celltype',
        ax=axs[i],
        use_raw=False,
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)
    plt.setp(axs[i].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
adata_filtered.write(output_path('02_SLC7A5_mono_Journal/01_Data/scRNA_adata_mono_cellannot_final.h5ad'))

# suppl Figure LAM markers

In [ ]:
adata_filtered=sc.read(artifact_path('02_SLC7A5_mono_Journal/01_Data/scRNA_adata_mono_cellannot_final.h5ad'))

In [ ]:
genes = ['TREM2','GPNMB', 'CD9', 'SPP1']
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(10, 3 * n_genes))

# genes가 1개일 때 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_filtered,
        keys=gene,
        groupby='celltype',
        ax=axs[i],
        use_raw=False,
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)
    plt.setp(axs[i].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/supplFig2A_violinplot.pdf'), bbox_inches='tight', dpi=300)

In [ ]:
genes = ['SLC7A5','CCR2', 'CD14', 'FCGR3A']
n_genes = len(genes)

fig, axs = plt.subplots(n_genes, 1, figsize=(10, 3 * n_genes))

# genes가 1개일 때 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_filtered,
        keys=gene,
        groupby='celltype',
        ax=axs[i],
        use_raw=False,
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)
    plt.setp(axs[i].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/supplFig2A_violinplot_2.pdf'), bbox_inches='tight', dpi=300)

In [ ]:
sc.pl.umap(
    adata_filtered,
    color='TREM2',
    #legend_loc='on data',
    legend_fontsize=6,
    frameon=False,
    title='Cell type prediction',
    size=10,
    show=False)
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2B.pdf'), bbox_inches='tight')
plt.close()

# Suppl Figure - SLC family

In [ ]:
deg_mono

In [ ]:
gene_set = {'SLC1A1','SLC1A2','SLC1A4','SLC1A5','SLC7A2','SLC7A5','SLC7A6','SLC7A7','SLC7A8','SLC7A9','SLC7A11',
         'SLC16A10','SLC25A12', 'SLC25A13','SLC25A18','SLC25A22','SLC25A44','SLC36A1','SLC36A4','SLC38A1',
         'SLC38A2','SLC38A3','SLC38A4','SLC38A6','SLC38A7','SLC38A9','SLC38A10','SLC43A1','SLC43A2'}

genes_in_data = [g for g in gene_set if g in adata.var_names]
print(f"Found {len(genes_in_data)} / {len(gene_set)} genes in adata")

expr = adata_filtered[:, genes_in_data].X
if hasattr(expr, 'toarray'):  # sparse matrix 처리
    expr = expr.toarray()

df = pd.DataFrame(expr, index=adata_filtered.obs_names, columns=genes_in_data)
df['celltype'] = adata_filtered.obs['celltype'].values
mean_expr = df.groupby('celltype')[genes_in_data].mean()

from scipy.stats import zscore
mean_expr_z = mean_expr.apply(zscore, axis=0)

# ── 방법 A: seaborn clustermap (계층적 클러스터링 포함) ──────────────────
sns.clustermap(
    mean_expr_z,
    cmap='RdBu_r',
    center=0,
    figsize=(10, 6),
    linewidths=0.5,
    annot=True,
    fmt='.2f',
    xticklabels=True,
    yticklabels=True,
)

In [ ]:
sc.pl.matrixplot(
    adata_filtered,
    var_names=genes_in_data,
    groupby='celltype',
    standard_scale='var',   # 유전자별 0-1 스케일
    cmap='Blues',
    colorbar_title='Scaled\nexpression',
    #save='_gene_set_heatmap.png',
)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Dotplot (ALL clusters from deg_cell) — Mean Expression + %Expressing
# ──────────────────────────────────────────────────────────────────────────

target_genes = ['SLC1A1','SLC1A2','SLC1A4','SLC1A5','SLC7A2','SLC7A5','SLC7A6','SLC7A7','SLC7A8','SLC7A9','SLC7A11',
         'SLC16A10','SLC25A12', 'SLC25A13','SLC25A18','SLC25A22','SLC25A44','SLC36A1','SLC36A4','SLC38A1',
         'SLC38A2','SLC38A3','SLC38A4','SLC38A6','SLC38A7','SLC38A9','SLC38A10','SLC43A1','SLC43A2']
clusters = sorted(deg_mono['group'].unique())

# cluster별 deg table dict화
deg_dict = {
    cl: deg_mono[deg_mono['group'] == cl].set_index('names')
    for cl in clusters
}

def get_raw_expr(adata, gene):
    if adata.raw is None:
        raise ValueError("adata.raw is None")
    if gene not in adata.raw.var_names:
        raise ValueError(f"{gene} not found in raw")
    idx = np.where(adata.raw.var_names == gene)[0][0]
    expr = adata.raw.X[:, idx]
    return expr.toarray().flatten() if sp.issparse(expr) else expr.flatten()

# ── 데이터 수집 ────────────────────────────────────────────────────────────
rows_g = []
for gene in target_genes:
    for cl in clusters:
        cl_label = f'Cluster {cl}'
        deg_sub = deg_dict[cl]

        # pct_nz_group
        pct_val = np.nan
        if gene in deg_sub.index and 'pct_nz_group' in deg_sub.columns:
            pct_val = float(deg_sub.loc[gene, 'pct_nz_group']) * 100

        # raw mean expression
        mask = adata_filtered.obs['celltype'] == cl
        if gene in adata_filtered.raw.var_names:
            expr_vals = get_raw_expr(adata_filtered, gene)[mask]
        else:
            expr_vals = np.zeros(mask.sum())
        logfc = np.nan
        if gene in deg_sub.index and 'logfoldchanges' in deg_sub.columns:
            logfc = float(deg_sub.loc[gene, 'logfoldchanges'])

        rows_g.append({
            'gene': gene,
            'cluster': cl,
            'cluster_id': cl,
            'mean_expr': float(expr_vals.mean()),
            'pct': pct_val,
            'logFC': logfc 
        })

df_g = pd.DataFrame(rows_g)
df_g['gene'] = pd.Categorical(df_g['gene'], categories=target_genes, ordered=True)

# ── Figure: Dotplot (axes swapped) ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(len(clusters) * 0.9 + 2, len(target_genes) * 0.9))

cluster_order = [f'{cl}' for cl in clusters]
gene_order = target_genes

cmap = plt.cm.Reds
vmin = 0
vmax = 2

for yi, gene in enumerate(gene_order):           # ← y: gene
    for xi, cl in enumerate(cluster_order):      # ← x: cluster
        row = df_g[(df_g['gene'] == gene) & (df_g['cluster'] == cl)]
        if len(row) == 0:
            continue

        pct_val = float(row['pct'].values[0]) if not pd.isna(row['pct'].values[0]) else 0.0
        #expr_val = float(row['mean_expr'].values[0]
        logfc_val = float(row['logFC'].values[0]) if not pd.isna(row['logFC'].values[0]) else 0.0
        logfc_val = np.clip(logfc_val, 0, 2)

        size = max(4, (pct_val / 100) * 800 * 1.2)
        alpha = 0.85 if pct_val > 0 else 0.2
        #color = cmap((expr_val - vmin) / (vmax - vmin + 1e-9))
        norm_val = (logfc_val - vmin) / (vmax - vmin + 1e-9)
        color = cmap(norm_val)

        ax.scatter(
            xi, yi,
            s=size,
            c=[color],
            edgecolors='grey',
            linewidths=0.5,
            alpha=alpha,
            zorder=3
        )

        ax.text(
            xi + 0.35, yi,                     # ← 핵심 수정
            f'{pct_val:.1f}%',
            ha='left',
            va='center',
            fontsize=8,
            color='#333333'
        )

# 축 설정 (swap 반영)
ax.set_xticks(range(len(cluster_order)))
ax.set_xticklabels(cluster_order, fontsize=10, rotation=45, ha='right')

ax.set_yticks(range(len(gene_order)))
ax.set_yticklabels(gene_order, fontsize=11, style='italic')

ax.set_xlim(-0.6, len(cluster_order) - 0.2)  # 오른쪽 공간 약간 확장
ax.set_ylim(-0.6, len(gene_order) - 0.4)
ax.invert_yaxis()

ax.set_title(
    'Amino acid transporter gene dotplot\n'
    '(Dot size = %Expressing, Dot color = Log2FC)',
    fontsize=11
)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# colorbar
sm = plt.cm.ScalarMappable(
    cmap=cmap,
    norm=plt.Normalize(vmin=vmin, vmax=vmax)
)
sm.set_array([])

cbar = fig.colorbar(sm, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label('log2 Fold Change (0–2)', fontsize=9)

# size legend
import matplotlib.lines as mlines
leg_handles = [
    mlines.Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor='#888888',
        markeredgecolor='grey',
        markersize=np.sqrt((pct_leg / 100) * 800),
        label=f'{pct_leg}%',
        alpha=0.7
    )
    for pct_leg in [10, 30, 50]
]
ax.grid(False)
ax.legend(
    handles=leg_handles,
    title='% Expressing',
    loc='lower right',
    fontsize=8,
    title_fontsize=8,
    framealpha=0.4
)

plt.tight_layout()
#plt.savefig(f"{FIG_DIR}/all_clusters_genes_dotplot.pdf", dpi=300, bbox_inches='tight')
plt.show()

pt = df_g.pivot_table(
    index='gene',
    columns='cluster',
    values=['mean_expr', 'pct'],
    sort=False
)

pt = pt.reindex(target_genes)

print(pt.round(3).to_string())

# Figure2D GSEA

In [ ]:
import gseapy as gp
gp.get_library_name() 

In [ ]:
adata=sc.read(artifact_path('02_SLC7A5_mono_Journal/01_Data/scRNA_adata_mono_cellannot_final.h5ad'))
adata

In [ ]:
# ── KEGG로 대사 pathway만 필터링해서 GSEA ──────────────────────────────
kegg_sets = gp.get_library('KEGG_2021_Human')

# 관심 pathway 키워드 필터링
#keywords = ['glycolysis', 'gluconeogenesis', 'citrate', 'oxidative phosphorylation',
#            'fatty acid degradation', 'fatty acid oxidation']

keywords = ['pi3k','mtor', 'hif', 'phagy', 'glycol', 'tca', 'oxidative','fatty' ]

kegg_metab = {
    k: v for k, v in kegg_sets.items()
    if any(kw.lower() in k.lower() for kw in keywords)
}

# 어떤 pathway가 걸렸는지 확인
for k in kegg_metab.keys():
    print(k)

In [ ]:
# ── KEGG로 대사 pathway만 필터링해서 GSEA ──────────────────────────────
kegg_sets = gp.get_library('KEGG_2021_Human')

# 관심 pathway 키워드 필터링
#keywords = ['glycolysis', 'gluconeogenesis', 'citrate', 'oxidative phosphorylation',
#            'fatty acid degradation', 'fatty acid oxidation']

keywords = ['phagy' ]

kegg_metab = {
    k: v for k, v in kegg_sets.items()
    if any(kw.lower() in k.lower() for kw in keywords)
}

# 어떤 pathway가 걸렸는지 확인
for k in kegg_metab.keys():
    print(k)

In [ ]:
import requests

def get_library_manual(name, organism='Human'):
    url = f"https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName={name}"
    response = requests.get(url)
    genesets = {}
    for line in response.text.split('\n'):
        if not line.strip():
            continue
        parts = line.strip().split('\t')
        k = parts[0]
        v = [x.split(',')[0] for x in parts[2:] if x]
        genesets[k] = v
    return genesets

wiki_sets = get_library_manual('WikiPathway_2023_Human')

In [ ]:
keywords = ['phagy' ]

wiki_metab = {
    k: v for k, v in wiki_sets.items()
    if any(kw.lower() in k.lower() for kw in keywords)
}

# 어떤 pathway가 걸렸는지 확인
for k in wiki_metab.keys():
    print(k)

In [ ]:
cluster_ids = sorted(adata.obs['celltype'].unique(),
                         key=lambda x: int(x) if str(x).isdigit() else x)
cluster_ids = [str(c) for c in cluster_ids]
print(f"Clusters to run: {cluster_ids}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Classical_monocyte vs SLC7A5_high_monocyte — Preranked GSEA
# ══════════════════════════════════════════════════════════════════════════

group_A = 'Classical_monocyte'
group_B = 'Special_monocyte'

# ── 1. 두 그룹만 subset ──────────────────────────────────────────────────
mask = adata.obs['celltype'].isin([group_A, group_B])
adata_sub = adata[mask].copy()

# ── 2. A vs B로 rank_genes_groups 새로 계산 ──────────────────────────────
sc.tl.rank_genes_groups(
    adata_sub,
    groupby='celltype',
    groups=[group_A],
    reference=group_B,
    method='wilcoxon',
    key_added='rank_AvsB',
)

# ── 3. Ranked list 추출 ───────────────────────────────────────────────────
rnk_df = sc.get.rank_genes_groups_df(
    adata_sub, group=group_A, key='rank_AvsB'
)
rnk = (rnk_df.set_index('names')['scores']
             .dropna()
             .sort_values(ascending=False))
rnk = rnk[~rnk.index.duplicated(keep='first')]

print(f"Ranked genes: {len(rnk)}")
print(f"Top 5:    {rnk.head().to_dict()}")
print(f"Bottom 5: {rnk.tail().to_dict()}")

In [ ]:
# 원하는 KEGG pathway 이름만 직접 지정
wanted_pathways = [
    "Oxidative phosphorylation",
    "PI3K-Akt signaling pathway",
    "HIF-1 signaling pathway",
    "mTOR signaling pathway",
    "Glycolysis / Gluconeogenesis",
    "Citrate cycle (TCA cycle)",
    "Fatty acid biosynthesis",
    "Fatty acid degradation"
]

kegg_sets = gp.get_library("KEGG_2021_Human")

kegg_selected = {
    k: v for k, v in kegg_sets.items()
    if k in wanted_pathways
}

print("사용할 KEGG pathways:")
for k in kegg_selected:
    print(f"  {k}")

In [ ]:
kegg_selected

In [ ]:
# rank_genes_groups 결과 dataframe

deg_df = sc.get.rank_genes_groups_df(

    adata_sub,

    group=group_A,

    key='rank_AvsB'

)

print(deg_df.columns)

print(deg_df.head())

In [ ]:
deg_sig = deg_df[

    (deg_df['pvals_adj'] < 0.05) &

    (deg_df['logfoldchanges'].abs() > 0.58)

]

print("DEG count:", len(deg_sig))

In [ ]:
# ── GSEA 실행 ─────────────────────────────────────────────────────────────
res_kegg = gp.prerank(
    rnk=rnk,
    gene_sets=kegg_selected,
    min_size=3,
    max_size=5000,
    permutation_num=1000,
    outdir=None,
    seed=42,
    verbose=False,
)

res_kegg_df = res_kegg.res2d.set_index('Term')
print(res_kegg_df[['NES','NOM p-val','FDR q-val','Tag %']].round(4).to_string())

# ── 시각화 ────────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

pathways = list(res_kegg_df.index)          # ← KEGG pathway 이름 그대로 사용

nes_vals  = [float(res_kegg_df.loc[p, 'NES'])       for p in pathways]
pval_vals = [float(res_kegg_df.loc[p, 'NOM p-val']) for p in pathways]  # ← res_kegg_df로 통일

colors = []
for nes, pval in zip(nes_vals, pval_vals):
    if pval < 0.05:
        colors.append('#d62728' if nes > 0 else '#1f77b4')
    elif pval < 0.25:
        colors.append('#f28e8e' if nes > 0 else '#90badb')
    else:
        colors.append('#cccccc')

fig, ax = plt.subplots(figsize=(8, len(pathways) * 0.6 + 1.5))  # ← pathway 수에 맞게 자동 조절
bars = ax.barh(pathways, nes_vals, color=colors, edgecolor='black', linewidth=0.7)

for i, (nes, pval) in enumerate(zip(nes_vals, pval_vals)):
    if np.isnan(nes):
        continue
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else \
          '†' if pval < 0.25 else 'ns'
    x_pos = nes + (0.05 if nes >= 0 else -0.05)
    ha = 'left' if nes >= 0 else 'right'
    ax.text(x_pos, i, sig, va='center', ha=ha, fontsize=11)

ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('NES (Normalized Enrichment Score)', fontsize=11)
ax.set_title(f'Metabolic GSEA (KEGG)\n{group_A}  vs  {group_B}', fontsize=12)

patches = [
    mpatches.Patch(color='#d62728', label=f'Higher in {group_A} (NOM p<0.05)'),
    mpatches.Patch(color='#1f77b4', label=f'Higher in {group_B} (NOM p<0.05)'),
    mpatches.Patch(color='#f28e8e', label='NOM p 0.05–0.25'),
    mpatches.Patch(color='#cccccc', label='ns'),
]
ax.legend(handles=patches, fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2Dv2_GSEA_Classical_vs_SLC7A5high_KEGG.pdf'), dpi=300)
plt.show()

In [ ]:
# leading edge 순서랑 ranked list에서 위치 확인
term = 'Autophagy'
result = res_kegg.results[term]

# ranked list에서 각 유전자 위치 확인
ledge_genes = result['lead_genes'].split(';')
for gene in ['MTOR', 'RPTOR', 'AKT1', 'AKT2', 'ULK1', 'BECN1', 'ATG7', 'SQSTM1']:
    if gene in rnk.index:
        rank_pos = rnk.rank(ascending=False)[gene]
        print(f"{gene}: rank {rank_pos:.0f}, score {rnk[gene]:.3f}")

In [ ]:
# leading edge 유전자 확인
autophagy_term = [t for t in res_kegg_df.index if 'autophagy' in t.lower() or 'Autophagy' in t]
print(autophagy_term)

for term in autophagy_term:
    print(f"\n{term}")
    print(res_kegg.results[term]['lead_genes'])

In [ ]:
# /home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/scripts/run_autophagy_gsea.py

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import gseapy as gp

# -----------------------------------------------------------------------------
# Input
# -----------------------------------------------------------------------------
FIG_DIR = output_dir('02_SLC7A5_mono_Journal/05_Figure')
os.makedirs(FIG_DIR, exist_ok=True)

# rnk:
# index = gene symbol, value = ranking score
# example:
# rnk = deg_df.set_index("gene")["score"].sort_values(ascending=False)

actual_autophagy_genes = [
    "ULK1", "BECN1", "ATG7", "ATG12", "ATG13",
    "WIPI1", "WIPI2", "ATG4B", 
    "GABARAP", "GABARAPL1", "ATG2A","SQSTM1"
]


group_A = "Classical_monocyte"
group_B = "SLC7A5_high_monocyte"

# -----------------------------------------------------------------------------
# Prepare ranked list
# -----------------------------------------------------------------------------
if isinstance(rnk, pd.DataFrame):
    if rnk.shape[1] != 1:
        raise ValueError("rnk DataFrame must have exactly one score column.")
    rnk = rnk.iloc[:, 0]

rnk = pd.Series(rnk).dropna()
rnk.index = rnk.index.astype(str).str.upper()
rnk = rnk[~rnk.index.duplicated(keep="first")]
rnk = rnk.sort_values(ascending=False)

# keep only genes present in ranked list
autophagy_genes_in_rnk = [g.upper() for g in actual_autophagy_genes if g.upper() in rnk.index]
missing_genes = sorted(set(g.upper() for g in actual_autophagy_genes) - set(autophagy_genes_in_rnk))

print(f"Autophagy genes in ranked list: {len(autophagy_genes_in_rnk)}/{len(actual_autophagy_genes)}")
if missing_genes:
    print("Missing genes:", ", ".join(missing_genes))

if len(autophagy_genes_in_rnk) < 3:
    raise ValueError("Too few autophagy genes found in rnk. Need at least 3 for current min_size.")

# -----------------------------------------------------------------------------
# Custom gene set
# -----------------------------------------------------------------------------
autophagy_sets = {
    "CUSTOM_AUTOPHAGY": autophagy_genes_in_rnk
}

# -----------------------------------------------------------------------------
# Run GSEA
# -----------------------------------------------------------------------------
res_autophagy = gp.prerank(
    rnk=rnk,
    gene_sets=autophagy_sets,
    min_size=3,
    max_size=5000,
    permutation_num=1000,
    outdir=None,
    seed=42,
    verbose=False,
)

res_autophagy_df = res_autophagy.res2d.set_index("Term")
print(res_autophagy_df[["NES", "NOM p-val", "FDR q-val", "Tag %"]].round(4).to_string())

# -----------------------------------------------------------------------------
# Bar plot
# -----------------------------------------------------------------------------
pathways = list(res_autophagy_df.index)
nes_vals = [float(res_autophagy_df.loc[p, "NES"]) for p in pathways]
pval_vals = [float(res_autophagy_df.loc[p, "NOM p-val"]) for p in pathways]

colors = []
for nes, pval in zip(nes_vals, pval_vals):
    if pval < 0.05:
        colors.append("#d62728" if nes > 0 else "#1f77b4")
    elif pval < 0.25:
        colors.append("#f28e8e" if nes > 0 else "#90badb")
    else:
        colors.append("#cccccc")

fig, ax = plt.subplots(figsize=(7, max(2.2, len(pathways) * 0.8 + 1.2)))
ax.barh(pathways, nes_vals, color=colors, edgecolor="black", linewidth=0.7)

for i, (nes, pval) in enumerate(zip(nes_vals, pval_vals)):
    if np.isnan(nes):
        continue
    sig = (
        "***" if pval < 0.001 else
        "**" if pval < 0.01 else
        "*" if pval < 0.05 else
        "†" if pval < 0.25 else
        "ns"
    )
    x_pos = nes + (0.05 if nes >= 0 else -0.05)
    ha = "left" if nes >= 0 else "right"
    ax.text(x_pos, i, sig, va="center", ha=ha, fontsize=11)

ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("NES (Normalized Enrichment Score)", fontsize=11)
ax.set_title(f"Autophagy GSEA\n{group_A} vs {group_B}", fontsize=12)

patches = [
    mpatches.Patch(color="#d62728", label=f"Higher in {group_A} (NOM p<0.05)"),
    mpatches.Patch(color="#1f77b4", label=f"Higher in {group_B} (NOM p<0.05)"),
    mpatches.Patch(color="#f28e8e", label="NOM p 0.05–0.25"),
    mpatches.Patch(color="#cccccc", label="ns"),
]
ax.legend(handles=patches, fontsize=8, loc="lower right")

plt.tight_layout()
plt.savefig(
    os.path.join(FIG_DIR, "Fig2Dv2_GSEA_Classical_vs_SLC7A5high_AUTOPHAGY_barplot.pdf"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()

# -----------------------------------------------------------------------------
# Enrichment curve
# -----------------------------------------------------------------------------
term = "CUSTOM_AUTOPHAGY"
if term in res_autophagy.results:
    gp.gseaplot(
        rank_metric=res_autophagy.ranking,
        term=term,
        **res_autophagy.results[term]
    )
    plt.suptitle(f"Autophagy GSEA\n{group_A} vs {group_B}", y=1.02, fontsize=12)
    plt.tight_layout()
    plt.savefig(
        os.path.join(FIG_DIR, "Fig2Dv3_GSEA_Classical_vs_SLC7A5high_AUTOPHAGY_curve.pdf"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

# -----------------------------------------------------------------------------
# Leading-edge / matched genes
# -----------------------------------------------------------------------------
print("\nMatched autophagy genes in ranked list:")
print(autophagy_genes_in_rnk)

if term in res_autophagy.results:
    lead_genes = res_autophagy.results[term].get("lead_genes", [])
    print("\nLeading-edge genes:")
    print(lead_genes)

In [ ]:
for gene in actual_autophagy_genes:
    if gene in rnk.index:
        print(f"{gene}: {rnk[gene]:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

wanted_terms = [
    "Fatty acid degradation",
    "Fatty acid biosynthesis",
    "Citrate cycle (TCA cycle)",
    "Glycolysis / Gluconeogenesis",
    "Oxidative phosphorylation",
    "mTOR signaling pathway",
    "HIF-1 signaling pathway",
    "Autophagy",
    "PI3K-Akt signaling pathway"
]

plot_df = res_kegg.res2d.set_index("Term").copy()
plot_df = plot_df.loc[plot_df.index.intersection(wanted_terms)].copy()
plot_df = plot_df.loc[wanted_terms]  # 원하는 순서 유지
plot_df = plot_df.dropna(subset=["NES", "NOM p-val"])

print(plot_df[["NES", "NOM p-val", "FDR q-val", "Tag %"]].round(4).to_string())

pathways = list(plot_df.index)
nes_vals = plot_df["NES"].astype(float).tolist()
pval_vals = plot_df["NOM p-val"].astype(float).tolist()

colors = []
for nes, pval in zip(nes_vals, pval_vals):
    if pval < 0.05:
        colors.append("#d62728" if nes > 0 else "#1f77b4")
    elif pval < 0.25:
        colors.append("#f28e8e" if nes > 0 else "#90badb")
    else:
        colors.append("#cccccc")

fig, ax = plt.subplots(figsize=(8, len(pathways) * 0.6 + 1.5))
ax.barh(pathways, nes_vals, color=colors, edgecolor="black", linewidth=0.7)

for i, (nes, pval) in enumerate(zip(nes_vals, pval_vals)):
    sig = (
        "***" if pval < 0.001 else
        "**" if pval < 0.01 else
        "*" if pval < 0.05 else
        "†" if pval < 0.25 else
        "ns"
    )
    x_pos = nes + (0.05 if nes >= 0 else -0.05)
    ha = "left" if nes >= 0 else "right"
    ax.text(x_pos, i, sig, va="center", ha=ha, fontsize=11)

ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("NES (Normalized Enrichment Score)", fontsize=11)
ax.set_title(f"Metabolic GSEA (KEGG)\n{group_A} vs {group_B}", fontsize=12)

patches = [
    mpatches.Patch(color="#d62728", label=f"Higher in {group_A} (NOM p<0.05)"),
    mpatches.Patch(color="#1f77b4", label=f"Higher in {group_B} (NOM p<0.05)"),
    mpatches.Patch(color="#f28e8e", label="NOM p 0.05–0.25"),
    mpatches.Patch(color="#cccccc", label="ns"),
]
ax.legend(handles=patches, fontsize=8, loc="lower right")

plt.tight_layout()
plt.savefig(
    output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2F_GSEA_Classical_vs_SLC7A5high_KEGG.pdf'),
    dpi=300,
)
plt.show()

In [ ]:
def sig_star(p):
    if np.isnan(p): return ""
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "\u2020" if p < 0.25 else ""

def fmt_pval(p):
    if np.isnan(p): return "NA"
    return f"{p:.2e}" if p < 0.001 else f"{p:.3f}"

metab_gs_names = list(kegg_metab.keys())
res_df = res_kegg.res2d.set_index('Term')

n_pw = len(metab_gs_names)
fig, axes = plt.subplots(1, n_pw, figsize=(n_pw * 5, 4.5),
                          constrained_layout=True)
if n_pw == 1:
    axes = [axes]

for ax, gs in zip(axes, metab_gs_names):
    r        = res_kegg.results.get(gs, {})
    es_curve = r.get('RES', None)

    if es_curve is None or len(es_curve) == 0:
        ax.set_title(gs + '\n(no result)', fontsize=10)
        continue

    nes_v  = float(res_df.loc[gs, 'NES'])       if gs in res_df.index else np.nan
    pval_v = float(res_df.loc[gs, 'NOM p-val']) if gs in res_df.index else np.nan

    # NES 부호에 따라 색 결정
    color = '#d62728' if nes_v >= 0 else '#1f77b4'   # red = Classical, blue = SLC7A5_high

    x_rank = np.linspace(0, 1, len(es_curve))
    ax.plot(x_rank, es_curve, color=color, lw=2.2, alpha=0.9)

    peak_idx = np.argmax(np.abs(es_curve))
    ax.scatter(x_rank[peak_idx], es_curve[peak_idx],
               color=color, s=40, zorder=5)

    # 히트 인덱스 tick marks
    hit_idx = (r.get('hit_indices') or r.get('hits') or r.get('hit_index') or [])
    for h in hit_idx:
        x_h = h / len(es_curve)
        ax.axvline(x_h, color=color, lw=0.5, alpha=0.3)

    stars = sig_star(pval_v)
    enriched_in = group_A if nes_v >= 0 else group_B
    lbl = (f"Enriched in: {enriched_in}\n"
           f"NES={nes_v:.2f}  NOM p={fmt_pval(pval_v)}"
           + (f"  {stars}" if stars else ""))

    ax.axhline(0, color='grey', lw=0.8, ls='--')
    ax.set_title(gs, fontsize=11, fontweight='bold')
    ax.set_xlabel('Gene rank (normalized)', fontsize=9)
    ax.set_ylabel('Running ES', fontsize=9)
    ax.legend(
        handles=[plt.Line2D([0], [0], color=color, lw=2, label=lbl)],
        fontsize=7.5, framealpha=0.35, loc='best'
    )
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.suptitle(
    f"GSEA Enrichment Score Curves (KEGG)\n"
    f"{group_A}  vs  {group_B}  [direct Wilcoxon z-score]",
    fontsize=12, fontweight='bold'
)
plt.savefig(
    output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2F_direct.pdf'),
    dpi=300, bbox_inches='tight'
)
plt.show()
print("ES curve plot (direct comparison) saved.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── 설정 ──────────────────────────────────────────────────────────────────
highlight_clusters = ["Classical_monocyte", "SLC7A5_high_monocyte"]
colors_map = {
    "Classical_monocyte":   "#d62728",
    "SLC7A5_high_monocyte": "#1f77b4",
}
FIG_DIR = output_dir('02_SLC7A5_mono_Journal/05_Figure')

def sig_star(p):
    if np.isnan(p): return ""
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "\u2020" if p < 0.25 else ""

def fmt_pval(p):
    if np.isnan(p): return "NA"
    return f"{p:.2e}" if p < 0.001 else f"{p:.3f}"

# ── 각 클러스터 vs Rest preranked GSEA 실행 ───────────────────────────────
gsea_results = {}

for cl in highlight_clusters:
    sc.tl.rank_genes_groups(
        adata,
        groupby="celltype",
        groups=[cl],
        reference="rest",
        method="wilcoxon",
        key_added=f"rank_{cl}",
    )
    rnk_cl = sc.get.rank_genes_groups_df(adata, group=cl, key=f"rank_{cl}")
    rnk_cl = (rnk_cl.set_index("names")["scores"]
                     .dropna()
                     .sort_values(ascending=False))
    rnk_cl = rnk_cl[~rnk_cl.index.duplicated(keep="first")]

    res = gp.prerank(
        rnk=rnk_cl,
        gene_sets=kegg_metab,
        min_size=3,
        max_size=5000,
        permutation_num=1000,
        outdir=None,
        seed=42,
        verbose=False,
    )
    gsea_results[cl] = res
    print(f"{cl}: GSEA done")

# ── metab_gs_names / nes_mat / pval_mat 구성 ─────────────────────────────
metab_gs_names = list(kegg_metab.keys())
cluster_ids    = highlight_clusters

nes_mat  = np.full((len(cluster_ids), len(metab_gs_names)), np.nan)
pval_mat = np.full((len(cluster_ids), len(metab_gs_names)), np.nan)

for ci, cl in enumerate(cluster_ids):
    df = gsea_results[cl].res2d.set_index("Term")
    for gi, gs in enumerate(metab_gs_names):
        if gs in df.index:
            nes_mat[ci, gi]  = float(df.loc[gs, "NES"])
            pval_mat[ci, gi] = float(df.loc[gs, "NOM p-val"])

# ── ES Curve 플롯 — 두 클러스터 overlay ──────────────────────────────────
n_pw = len(metab_gs_names)
fig, axes = plt.subplots(1, n_pw, figsize=(n_pw * 5, 4.5),
                          constrained_layout=True)
if n_pw == 1:
    axes = [axes]

for ax, gs in zip(axes, metab_gs_names):
    gi = metab_gs_names.index(gs)
    legend_handles = []

    for cl in highlight_clusters:
        ci = cluster_ids.index(cl)

        r = gsea_results[cl].results.get(gs, {})
        es_curve = r.get("RES", None)
        if es_curve is None or len(es_curve) == 0:
            continue

        nes_v  = nes_mat[ci, gi]
        pval_v = pval_mat[ci, gi]
        x_rank = np.linspace(0, 1, len(es_curve))
        color  = colors_map[cl]

        ax.plot(x_rank, es_curve, color=color, lw=2.2, alpha=0.9)

        peak_idx = np.argmax(np.abs(es_curve))
        ax.scatter(x_rank[peak_idx], es_curve[peak_idx],
                   color=color, s=40, zorder=5)

        stars = sig_star(pval_v)
        lbl = (f"{cl}\nNES={nes_v:.2f}  NOM p={fmt_pval(pval_v)}"
               + (f" {stars}" if stars else ""))
        legend_handles.append(
            plt.Line2D([0], [0], color=color, lw=2, label=lbl)
        )

    ax.axhline(0, color="grey", lw=0.8, ls="--")
    ax.set_title(gs, fontsize=12, fontweight="bold")
    ax.set_xlabel("Gene rank (normalized)", fontsize=9)
    ax.set_ylabel("Running ES", fontsize=9)
    ax.legend(handles=legend_handles, fontsize=7, framealpha=0.35,
              loc="best", title_fontsize=7)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    "GSEA Enrichment Score Curves\nClassical_monocyte  vs  SLC7A5_high_monocyte (each vs Rest)",
    fontsize=12, fontweight="bold"
)
plt.savefig(output_path('02_SLC7A5_mono_Journal/05_Figure/Fig2F_GSEAeach.pdf'), dpi=300)
plt.show()
print("ES curve plot saved.")


# Figure2 classical vs special

In [ ]:
adata=sc.read(artifact_path('02_SLC7A5_mono_Journal/01_Data/scRNA_adata_mono_cellannot_final.h5ad'))

In [ ]:
adata.obs['celltype'].value_counts()

In [ ]:
sc.pl.umap(adata, color=['celltype'])

In [ ]:
sc.pl.umap(adata, color=['state'])

In [ ]:
import scanpy as sc
import pandas as pd

GROUP_COL = "celltype"
GROUP_1 = "Classical_monocyte"
GROUP_2 = "Special_monocyte"

adata_mono = adata[adata.obs[GROUP_COL].isin([GROUP_1, GROUP_2])].copy()
adata_mono.obs[GROUP_COL] = pd.Categorical(
    adata_mono.obs[GROUP_COL],
    categories=[GROUP_1, GROUP_2]
)

sc.tl.rank_genes_groups(
    adata_mono,
    groupby=GROUP_COL,
    method="wilcoxon",
    use_raw=True,
    pts=True
)

deg_all = sc.get.rank_genes_groups_df(adata_mono, group=None).copy()

deg_classical_vs_special = deg_all[deg_all["group"] == GROUP_1].copy()
deg_special_vs_classical = deg_all[deg_all["group"] == GROUP_2].copy()

deg_classical_vs_special["comparison"] = f"{GROUP_1}_vs_{GROUP_2}"
deg_special_vs_classical["comparison"] = f"{GROUP_2}_vs_{GROUP_1}"

deg_classical_vs_special.to_csv(output_path('02_SLC7A5_mono_Journal/05_Figure/DEG_Classical_monocyte_vs_Special_monocyte.csv'), index=False)
deg_special_vs_classical.to_csv(output_path('02_SLC7A5_mono_Journal/05_Figure/DEG_Special_monocyte_vs_Classical_monocyte.csv'), index=False)

print(deg_classical_vs_special.columns.tolist())
print(deg_classical_vs_special.head())